# 01 · RAG 파이프라인 기본 구성

Phase 3 — `data/docs/` 코퍼스를 청킹·임베딩하여 **pgvector**에 적재하고, Ollama LLM + LCEL 체인으로 end-to-end 응답까지 확인한다.

주요 로직은 `src/`(chunker·embedder·retriever·rag)로 분리했다. 이 노트북은 **소량(MAX_DOCS)으로 단위 확인**하는 용도다.

> 전체 1,417개 인덱싱처럼 무거운 작업은 노트북이 아니라 `python scripts/index_corpus.py`로 실행한다.

> 사전 준비: `docker compose up -d` (pgvector healthy), `ollama pull gemma4:e2b`, `.env` 작성

## 0. 설정

경로·하이퍼파라미터·접속정보는 `src/config.py`에 모여 있다. 여기선 빠른 확인용 `MAX_DOCS`만 정한다.

In [ ]:
from src import chunker, config, dataset, embedder, rag, retriever

# 빠른 검증용: 일부 문서만 인덱싱 (전체는 scripts/index_corpus.py 사용)
MAX_DOCS = 200

golden_set = dataset.load_golden_set()
print(f"골든셋: {len(golden_set)}개 / 임베딩: {config.EMBED_MODEL} / 컬렉션: {config.COLLECTION_NAME}")

## 1. 문서 로드 & 청킹

In [ ]:
documents = chunker.load_documents(max_docs=MAX_DOCS)
chunks = chunker.chunk_documents(documents)

lengths = [len(c.page_content) for c in chunks]
print(f"로드 문서: {len(documents)}개 (MAX_DOCS={MAX_DOCS}) / 청크: {len(chunks)}개")
print(f"청크 길이 — min {min(lengths)} / 평균 {sum(lengths)//len(lengths)} / max {max(lengths)}")
print('예시 청크 메타:', chunks[0].metadata)
print(chunks[0].page_content[:200], '...')

## 2. 임베딩 모델 준비

`dragonkue/BGE-m3-ko` 로드 (정규화 + device 자동 선택).

In [ ]:
embeddings = embedder.build_embeddings()
probe = embeddings.embed_query('임베딩 차원 확인용 테스트 문장')
print(f"device={embedder.detect_device()} / 임베딩 차원: {len(probe)}")

## 3. pgvector 적재 & 유사도 검색

테이블을 (재)생성하고 청크를 적재한다. `index_chunks`가 tqdm으로 진행도를 표시한다.

> ⚠️ `init_table(overwrite_existing=True)`는 기존 데이터를 지운다. 전체 코퍼스 적재는 `python scripts/index_corpus.py`를 쓰는 게 낫다.

In [ ]:
engine = retriever.get_engine()
retriever.init_table(engine, config.COLLECTION_NAME, vector_size=len(probe), overwrite_existing=True)
store = retriever.get_store(engine, embeddings)

n = retriever.index_chunks(store, chunks)  # tqdm 진행도
print(f"적재 완료: {n}개 청크 → 테이블 {config.COLLECTION_NAME!r}")

In [ ]:
query = golden_set[0]['question']
hits = store.similarity_search_with_score(query, k=3)
print(f"질문: {query}")
print(f"정답: {golden_set[0]['ground_truth']}")
print('=' * 70)
for rank, (doc, score) in enumerate(hits, 1):
    preview = doc.page_content[:160].replace(chr(10), ' ')
    print(f"[{rank}] score={score:.4f}  title={doc.metadata.get('title')}")
    print(preview, '...')
    print('-' * 70)

## 4–5. Ollama LLM + RAG 체인 (LCEL)

pgvector retriever + Ollama `gemma4:e2b`를 LCEL로 묶는다. 검색 근거(`source_documents`)도 함께 반환한다.

In [ ]:
retr = retriever.get_retriever(store, k=4)
llm = rag.build_llm()
print('Ollama 연결 확인:', llm.invoke('한 문장으로 자기소개해줘.').content[:60])

qa_chain = rag.build_qa_chain(retr, llm)
print('RAG 체인 구성 완료 (LCEL, retriever k=4, stuff)')

In [ ]:
sample = golden_set[0]
result = qa_chain.invoke(sample['question'])
print('질문      :', sample['question'])
print('정답      :', sample['ground_truth'])
print('RAG 응답  :', result['answer'].strip())
print('=' * 70)
print('근거 문서:')
for i, doc in enumerate(result['source_documents'], 1):
    print(f"  [{i}] {doc.metadata.get('title')} — {doc.page_content[:80].strip()}...")

## 정리

- `data/docs/` → 청킹 → BGE-m3-ko 임베딩 → pgvector 적재 → LCEL RAG 체인 end-to-end 확인
- **다음(Phase 4)**: `02_ragas_eval.ipynb`로 RAGAS 4대 지표 평가

> 제대로 된 baseline은 전체 코퍼스 인덱싱이 필요하다: `python scripts/index_corpus.py` (MAX_DOCS 제한 없음)